# 01 — Data download and cleaning

Downloads and caches the raw UCI CSV, resamples to hourly, and inspects the
result. All logic lives in `src/appliance_energy/data.py`; this notebook only
calls it and displays output.

**Report sections fed:** 2 (Data and preprocessing).


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, stationarity
from appliance_energy.models import benchmarks, feature_models, foundation, sarimax

pd.set_option("display.width", 140)


## Raw ten-minute data

In [ ]:
raw = data.load_raw()
print(raw.shape)
print(raw.index.min(), "->", raw.index.max())
raw.head()


### Completeness of the ten-minute index

In [ ]:
expected = pd.date_range(raw.index.min(), raw.index.max(), freq="10min")
missing = expected.difference(raw.index)
print(f"expected {len(expected)}, present {len(raw)}, missing {len(missing)}")


### Correlation of `lights` with the target\n\nJustifies the exclusion described in report Section 2.

In [ ]:
print(raw[[config.TARGET, "lights"]].corr().iloc[0, 1].round(3))


## Hourly resampling\n\nNote the units: the mean gives average Wh per ten-minute interval within the hour.

In [ ]:
frame = data.load_hourly()
y = frame[config.TARGET]

y_train, y_test = data.train_test_split(y)
test_index = y_test.index

print(f"train {y_train.index.min()} -> {y_train.index.max()}  ({len(y_train)})")
print(f"test  {test_index.min()} -> {test_index.max()}  ({len(y_test)})")


In [ ]:
print(frame.shape)
frame[config.TARGET].describe().round(2)


### Distributional shape\n\nRecord these for report Section 3.1.

In [ ]:
print(f"mean      {y.mean():.2f}")
print(f"median    {y.median():.2f}")
print(f"std       {y.std():.2f}")
print(f"skewness  {y.skew():.2f}")
print(f"p95/median {y.quantile(0.95) / y.median():.2f}")
